# Projeto Integrador — Bloco 3
## Relatório Visual de um Voo de Fiscalização

**Bloco 3 — Matplotlib + Seaborn**  
**Projeto:** IC-FAPEMA — Detecção de capacetes em motociclistas via drone  
**Objetivo:** Gerar um relatório completo de um voo: bounding boxes, curvas de treinamento e avaliação do modelo

---

Neste projeto você vai simular o fluxo completo de análise de um voo de fiscalização:  
1. Visualizar as detecções do YOLO em um frame do drone  
2. Analisar as curvas de treinamento do modelo  
3. Avaliar a qualidade do modelo com a matriz de confusão  
4. Gerar um relatório final com todas as métricas

---

In [ ]:
# Importar todas as bibliotecas do Bloco 3
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import patches
import seaborn as sns

---
## Parte 1 — Visualizar o frame do drone
*(Aula 3.1 — Matplotlib + bounding boxes)*

O drone capturou um frame com 6 motociclistas.  
O YOLO detectou todos e retornou as coordenadas normalizadas abaixo.

In [ ]:
# ── Dados do frame ────────────────────────────────────────────────────────────
# Imagem simulada do drone — fundo cinza escuro 800x600
imagem = np.ones((600, 800, 3), dtype=np.uint8) * 60

# Detecções do YOLO — classe 0 = com capacete | classe 1 = sem capacete
deteccoes = [
    {'classe': 0, 'x_centro': 0.15, 'y_centro': 0.25, 'largura': 0.12, 'altura': 0.18},
    {'classe': 1, 'x_centro': 0.40, 'y_centro': 0.35, 'largura': 0.14, 'altura': 0.20},
    {'classe': 0, 'x_centro': 0.65, 'y_centro': 0.30, 'largura': 0.11, 'altura': 0.17},
    {'classe': 1, 'x_centro': 0.25, 'y_centro': 0.65, 'largura': 0.13, 'altura': 0.19},
    {'classe': 0, 'x_centro': 0.55, 'y_centro': 0.70, 'largura': 0.12, 'altura': 0.18},
    {'classe': 1, 'x_centro': 0.80, 'y_centro': 0.60, 'largura': 0.14, 'altura': 0.21},
]

img_largura = imagem.shape[1]
img_altura  = imagem.shape[0]

# ── Visualização ──────────────────────────────────────────────────────────────
fig, ax = plt.subplots(1, figsize=(11, 7))
ax.imshow(imagem)

infratores = 0

for det in deteccoes:
    x_pixel = (det['x_centro'] - det['largura']  / 2) * img_largura
    y_pixel = (det['y_centro'] - det['altura']   / 2) * img_altura
    w_pixel =  det['largura']  * img_largura
    h_pixel =  det['altura']   * img_altura

    cor   = 'red'   if det['classe'] == 1 else 'lime'
    label = 'SEM capacete ⚠' if det['classe'] == 1 else 'com capacete'

    retangulo = patches.Rectangle(
        (x_pixel, y_pixel), w_pixel, h_pixel,
        linewidth=2, edgecolor=cor, facecolor='none'
    )
    ax.add_patch(retangulo)
    ax.text(x_pixel, y_pixel - 8, label, color=cor, fontsize=9, weight='bold')

    if det['classe'] == 1:
        infratores += 1

ax.set_title(f'Frame do drone — {len(deteccoes)} motociclistas detectados | {infratores} sem capacete',
             fontsize=12, weight='bold', color='white')
ax.set_facecolor('#3c3c3c')
ax.axis('off')
plt.tight_layout()
plt.show()

print(f"Total detectado: {len(deteccoes)} motociclistas")
print(f"Com capacete:    {len(deteccoes) - infratores}")
print(f"Sem capacete:    {infratores} ← infratores")

---
## Parte 2 — Curvas de treinamento do modelo
*(Aula 3.2 — Curvas de loss e accuracy)*

Antes deste voo, o modelo foi treinado por 15 épocas.  
Analise as curvas e verifique se o modelo treinou bem.

In [ ]:
# ── Dados do treinamento ──────────────────────────────────────────────────────
epochs = list(range(1, 16))

loss_treino = [0.98, 0.84, 0.72, 0.61, 0.52, 0.44, 0.38, 0.33, 0.29, 0.26, 0.23, 0.21, 0.19, 0.18, 0.17]
loss_val    = [1.00, 0.87, 0.75, 0.65, 0.57, 0.50, 0.45, 0.41, 0.38, 0.36, 0.34, 0.33, 0.32, 0.32, 0.32]

acc_treino  = [0.42, 0.55, 0.64, 0.71, 0.76, 0.80, 0.83, 0.86, 0.88, 0.90, 0.91, 0.93, 0.94, 0.95, 0.96]
acc_val     = [0.40, 0.52, 0.61, 0.68, 0.73, 0.77, 0.80, 0.82, 0.84, 0.85, 0.86, 0.87, 0.87, 0.87, 0.87]

# ── Gráficos ──────────────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Loss
ax1.plot(epochs, loss_treino, label='Treino',    color='blue',   linewidth=2)
ax1.plot(epochs, loss_val,    label='Validação', color='orange', linewidth=2, linestyle='--')
ax1.set_title('Curva de Loss')
ax1.set_xlabel('Épocas')
ax1.set_ylabel('Loss (erro)')
ax1.legend()

# Accuracy
ax2.plot(epochs, acc_treino, label='Treino',    color='blue',   linewidth=2)
ax2.plot(epochs, acc_val,    label='Validação', color='orange', linewidth=2, linestyle='--')
ax2.set_title('Curva de Accuracy')
ax2.set_xlabel('Épocas')
ax2.set_ylabel('Accuracy (acurácia)')
ax2.legend()

fig.suptitle('Treinamento YOLO — Modelo de detecção de capacetes', fontsize=13, weight='bold')
plt.tight_layout()
plt.show()

print(f"Melhor accuracy de validação: {max(acc_val):.0%} na época {acc_val.index(max(acc_val))+1}")

---
## Parte 3 — Avaliação do modelo
*(Aula 3.3 — Matriz de confusão e Precision-Recall)*

O modelo foi avaliado em 300 imagens de teste.  
A matriz abaixo resume os resultados.

In [ ]:
# ── Matriz de confusão ────────────────────────────────────────────────────────
matriz = np.array([
    [185, 12],   # real: com capacete
    [  8, 95],   # real: sem capacete
])

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    matriz, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Com capacete', 'Sem capacete'],
    yticklabels=['Com capacete', 'Sem capacete'],
    ax=ax
)
ax.set_xlabel('Previsto pelo modelo', fontsize=11)
ax.set_ylabel('Real', fontsize=11)
ax.set_title('Matriz de Confusão — Avaliação do modelo', fontsize=12, weight='bold')
plt.tight_layout()
plt.show()

# ── Métricas ──────────────────────────────────────────────────────────────────
VN = matriz[0][0]
FP = matriz[0][1]
FN = matriz[1][0]
VP = matriz[1][1]

precision = VP / (VP + FP)
recall    = VP / (VP + FN)
total     = matriz.sum()
acertos   = VN + VP

print("=" * 45)
print("       RELATÓRIO DE AVALIAÇÃO DO MODELO")
print("=" * 45)
print(f"Total de imagens avaliadas: {total}")
print(f"Acertos:   {acertos} ({acertos/total*100:.1f}%)")
print(f"Erros:     {total-acertos} ({(total-acertos)/total*100:.1f}%)")
print("-" * 45)
print(f"FN (infratores ignorados): {FN}  ← erro mais grave")
print(f"FP (falsos alarmes):       {FP}")
print("-" * 45)
print(f"Precision: {precision:.2f}  ({precision*100:.1f}% dos alarmes são reais)")
print(f"Recall:    {recall:.2f}  ({recall*100:.1f}% dos infratores capturados)")
print("=" * 45)

---
## Resumo do Projeto Integrador

| Parte | Ferramenta | O que foi feito |
|---|---|---|
| Frame do drone | Matplotlib + patches | Bounding boxes coloridas por classe |
| Treinamento | Matplotlib subplots | Curvas de loss e accuracy lado a lado |
| Avaliação | Seaborn heatmap | Matriz de confusão com Precision e Recall |

### Conclusão do Bloco 3

Você agora sabe **visualizar** todo o ciclo do projeto:
- Ver as detecções do drone em cima da imagem real
- Acompanhar se o modelo está aprendendo corretamente
- Avaliar a qualidade final com métricas reais de visão computacional

**Próximo passo → Bloco 4: OpenCV** — carregar e manipular imagens reais do drone 🚁